# Proposed Model: LightGBM
**DSC 148 Final Project**  

LightGBM is a gradient-boosted tree ensemble with three advantages over the baselines:
1. **Handles NaN natively** -- no imputation needed; missingness in V-features is itself informative.
2. **Captures nonlinear interactions** -- fraud signal lies in combinations of card, device, and behavioral features that a linear model cannot express.
3. **Uses all 450+ features** -- V1--V339 (excluded from baselines due to high missingness) are included.

> **Run `02_baseline_lr.ipynb` and `03_baseline_nb.ipynb` first** to generate `results/lr_metrics.json`, `results/nb_metrics.json`, `results/lr_probs.npy`, `results/nb_probs.npy` used in the comparison section.

In [ ]:
#!pip install pandas numpy matplotlib seaborn scikit-learn lightgbm -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
SEED = 42
print('Libraries loaded.')

## 1. Data Loading and Preprocessing

Identical clean pipeline used in notebooks 02 and 03.

In [ ]:
def reduce_mem(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    return df

train_trans = reduce_mem(pd.read_csv('data/train_transaction.csv'))
train_id    = reduce_mem(pd.read_csv('data/train_identity.csv'))
df = train_trans.merge(train_id, on='TransactionID', how='left')

START_DATE = pd.Timestamp('2017-11-30')
dt = START_DATE + pd.to_timedelta(df['TransactionDT'], unit='s')
df['tx_hour']  = dt.dt.hour.astype('int8')
df['tx_dow']   = dt.dt.dayofweek.astype('int8')
df['tx_month'] = dt.dt.month.astype('int8')
df['card_age'] = (df['TransactionDT'] // 86400 - df['D1']).astype('float32')
print(f'Merged shape: {df.shape}')

In [ ]:
df_sorted = df.sort_values('TransactionDT').reset_index(drop=True)
n       = len(df_sorted)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_df = df_sorted.iloc[:n_train].copy()
val_df   = df_sorted.iloc[n_train:n_train+n_val].copy()
test_df  = df_sorted.iloc[n_train+n_val:].copy()

y_train = train_df['isFraud'].values
y_val   = val_df['isFraud'].values
y_test  = test_df['isFraud'].values

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')
print(f'Fraud -- train : {y_train.mean()*100:.2f}%  val : {y_val.mean()*100:.2f}%  test : {y_test.mean()*100:.2f}%')

CAT_COLS = [
    'ProductCD','card4','card6','P_emaildomain','R_emaildomain',
    'M1','M2','M3','M4','M5','M6','M7','M8','M9',
    'DeviceType','DeviceInfo',
    'id_12','id_15','id_16','id_23','id_27','id_28',
    'id_29','id_30','id_31','id_33','id_34','id_35','id_36','id_37','id_38'
]
CAT_COLS = [c for c in CAT_COLS if c in train_df.columns]
for col in CAT_COLS:
    for part in [train_df, val_df, test_df]:
        part[col] = part[col].fillna('unknown').astype(str)
    cats = train_df[col].unique().tolist()
    for part in [train_df, val_df, test_df]:
        part[col] = pd.Categorical(part[col], categories=cats).codes
print(f'Label-encoded {len(CAT_COLS)} columns.')

In [ ]:
for part in [train_df, val_df, test_df]:
    part['uid'] = (part['card1'].astype(str) + '_' +
                   part['addr1'].fillna(-1).astype(int).astype(str) + '_' +
                   part['card_age'].fillna(-1).round(0).astype(int).astype(str))

c_cols    = [f'C{i}' for i in range(1, 15)]
uid_stats = (train_df.groupby('uid')
               .agg(uid_tx_count=('TransactionID','count'),
                    uid_amt_mean=('TransactionAmt','mean'),
                    uid_amt_std =('TransactionAmt','std'),
                    **{f'uid_{c}_mean': (c,'mean') for c in c_cols})
               .reset_index())
fallback = {
    'uid_tx_count': 1,
    'uid_amt_mean': float(train_df['TransactionAmt'].mean()),
    'uid_amt_std' : float(train_df['TransactionAmt'].std()),
    **{f'uid_{c}_mean': float(train_df[c].mean()) for c in c_cols},
}

def attach_uid(df_part, stats, fb):
    out = df_part.merge(stats, on='uid', how='left')
    for col, val in fb.items():
        out[col] = out[col].fillna(val).astype('float32')
    return out.drop(columns=['uid'])

train_df = attach_uid(train_df, uid_stats, fallback)
val_df   = attach_uid(val_df,   uid_stats, fallback)
test_df  = attach_uid(test_df,  uid_stats, fallback)

DROP = ['TransactionID','isFraud','TransactionDT']
FULL_FEATURES = [c for c in train_df.columns if c not in DROP]

def _full(df_part):
    X = df_part[FULL_FEATURES].copy()
    X['TransactionAmt'] = np.log1p(X['TransactionAmt'])
    return X

X_train_f = _full(train_df)
X_val_f   = _full(val_df)
X_test_f  = _full(test_df)

print(f'Full features : {len(FULL_FEATURES)}')
print(f'Train : {X_train_f.shape[0]:,}  |  Val : {X_val_f.shape[0]:,}  |  Test : {X_test_f.shape[0]:,}')

## 2. LightGBM

`scale_pos_weight = n_legitimate / n_fraud` upweights the fraud class in the loss.  
Early stopping on **val AUC** (not logloss -- logloss is minimised by pushing all probabilities near zero under heavy imbalance).  
Threshold tuned on **val F1**; test set touched once.

In [ ]:
neg, pos = np.bincount(y_train)
spw = neg / pos
print(f'scale_pos_weight = {spw:.1f}  ({neg:,} legitimate / {pos:,} fraud)')

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05, num_leaves=64, max_depth=8,
    scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.3,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_model.fit(
    X_train_f, y_train,
    eval_set=[(X_val_f, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)

y_prob_lgb_val = lgb_model.predict_proba(X_val_f)[:, 1]
y_prob_lgb     = lgb_model.predict_proba(X_test_f)[:, 1]

print(f'\nProb range (val) : [{y_prob_lgb_val.min():.4f}, {y_prob_lgb_val.max():.4f}]')
print(f'Prob range (test): [{y_prob_lgb.min():.4f}, {y_prob_lgb.max():.4f}]')

thr_lgb  = np.linspace(y_prob_lgb_val.min()+1e-4, y_prob_lgb_val.max()-1e-4, 300)
best_lgb = float(thr_lgb[int(np.argmax(
    [f1_score(y_val, (y_prob_lgb_val>=t).astype(int), zero_division=0) for t in thr_lgb]
))])
y_pred_lgb = (y_prob_lgb >= best_lgb).astype(int)
print(f'Optimal threshold (val): {best_lgb:.4f}')

print('\n' + '='*35)
print('LightGBM -- Test Set')
print('='*35)
print(classification_report(y_test, y_pred_lgb, target_names=['Legitimate','Fraud']))
print(f'AUC-ROC : {roc_auc_score(y_test, y_prob_lgb):.4f}')

## 3. Feature Importance

In [ ]:
importances = pd.Series(lgb_model.feature_importances_, index=FULL_FEATURES).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
importances.head(25).sort_values().plot(kind='barh', ax=ax, color='#4878CF')
ax.set_title('Fig. 1: LightGBM Feature Importance -- Top 25 (split count)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 10 features:')
for rank, (feat, score) in enumerate(importances.head(10).items(), 1):
    print(f'  {rank:2d}. {feat:<30s}  {score}')

## 4. Ablation Study -- Effect of Identity Features

Retrain on transaction features only (no `id_*`, `DeviceType`, `DeviceInfo`) to quantify the identity table's contribution.

In [ ]:
ID_COLS = ([c for c in FULL_FEATURES if c.startswith('id_')] +
           [c for c in FULL_FEATURES if c in ('DeviceType','DeviceInfo')])
TX_ONLY = [c for c in FULL_FEATURES if c not in ID_COLS]

lgb_noid = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.05, num_leaves=64, max_depth=8,
    scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=0.3,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgb_noid.fit(
    X_train_f[TX_ONLY], y_train,
    eval_set=[(X_val_f[TX_ONLY], y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

y_prob_noid_val = lgb_noid.predict_proba(X_val_f[TX_ONLY])[:, 1]
y_prob_noid     = lgb_noid.predict_proba(X_test_f[TX_ONLY])[:, 1]

thr_noid  = np.linspace(y_prob_noid_val.min()+1e-4, y_prob_noid_val.max()-1e-4, 300)
best_noid = float(thr_noid[int(np.argmax(
    [f1_score(y_val, (y_prob_noid_val>=t).astype(int), zero_division=0) for t in thr_noid]
))])
y_pred_noid = (y_prob_noid >= best_noid).astype(int)

def get_metrics(y_true, y_pred, y_prob):
    return {
        'Accuracy'          : round(accuracy_score(y_true, y_pred), 4),
        'Precision (Fraud)' : round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall (Fraud)'    : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1 (Fraud)'        : round(f1_score(y_true, y_pred, zero_division=0), 4),
        'AUC-ROC'           : round(roc_auc_score(y_true, y_prob), 4),
    }

ablation_df = pd.DataFrame({
    'LightGBM (tx only)' : get_metrics(y_test, y_pred_noid, y_prob_noid),
    'LightGBM (+identity)': get_metrics(y_test, y_pred_lgb,  y_prob_lgb),
}).T

print('Table 1: Ablation -- Identity Feature Contribution')
print(ablation_df.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ablation_df[['F1 (Fraud)','AUC-ROC']].plot(kind='bar', ax=ax,
    color=['#D65F5F','#4878CF'], edgecolor='white')
ax.set_title('Fig. 2: Ablation -- Transaction-Only vs. +Identity Features')
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=10)
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Results Comparison

Loads saved metrics from `02_baseline_lr.ipynb` and `03_baseline_nb.ipynb`.  
If those files do not exist, run the baseline notebooks first.

In [ ]:
try:
    with open('results/lr_metrics.json') as f: lr_m = json.load(f)
    with open('results/nb_metrics.json') as f: nb_m = json.load(f)
    y_prob_lr = np.load('results/lr_probs.npy')
    y_prob_nb = np.load('results/nb_probs.npy')
    has_baselines = True
    print('Baseline metrics loaded.')
except FileNotFoundError as e:
    print(f'Missing file: {e}')
    print('Run 02_baseline_lr.ipynb and 03_baseline_nb.ipynb first.')
    has_baselines = False

lgb_metrics = get_metrics(y_test, y_pred_lgb, y_prob_lgb)

if has_baselines:
    results = pd.DataFrame({
        'Logistic Regression': {
            'Accuracy'         : lr_m['accuracy'],
            'Precision (Fraud)': lr_m['precision'],
            'Recall (Fraud)'   : lr_m['recall'],
            'F1 (Fraud)'       : lr_m['f1'],
            'AUC-ROC'          : lr_m['auc'],
        },
        'Naive Bayes': {
            'Accuracy'         : nb_m['accuracy'],
            'Precision (Fraud)': nb_m['precision'],
            'Recall (Fraud)'   : nb_m['recall'],
            'F1 (Fraud)'       : nb_m['f1'],
            'AUC-ROC'          : nb_m['auc'],
        },
        'LightGBM': lgb_metrics,
    }).T
    print('\nTable 2: Model Performance Comparison')
    print(results.to_string())
else:
    print('\nLightGBM metrics only (baselines not loaded):')
    print(pd.DataFrame([lgb_metrics]).to_string())

In [ ]:
if has_baselines:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ROC curves
    for name, yp, col in [('Logistic Regression', y_prob_lr, '#F0A202'),
                           ('Naive Bayes',          y_prob_nb, '#D65F5F'),
                           ('LightGBM',             y_prob_lgb,'#4878CF')]:
        fpr, tpr, _ = roc_curve(y_test, yp)
        auc = roc_auc_score(y_test, yp)
        axes[0].plot(fpr, tpr, color=col, label=f'{name}  (AUC={auc:.3f})')
    axes[0].plot([0,1],[0,1],'k--',linewidth=1,label='Random')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('Fig. 3: ROC Curves')
    axes[0].legend(loc='lower right')

    # Bar chart comparison
    results[['F1 (Fraud)','AUC-ROC','Recall (Fraud)','Precision (Fraud)']].plot(
        kind='bar', ax=axes[1], colormap='tab10', edgecolor='white', width=0.7)
    axes[1].set_title('Fig. 4: Model Performance Comparison')
    axes[1].set_ylabel('Score')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    axes[1].set_ylim(0, 1.1)
    axes[1].legend(loc='lower right')

    plt.tight_layout()
    plt.show()

## 6. Conclusions

- **Class imbalance** (~3.5% fraud) makes accuracy a misleading metric; AUC-ROC and F1 on the fraud class are the primary evaluation criteria.
- **Logistic Regression** establishes a linear baseline. Its assumptions (linear separability, no feature interactions) are violated by the card/device/behavioral interaction structure of fraud.
- **Naive Bayes** assumes conditional independence across features, which is violated by correlated C1--C14 and card identifier interactions. This produces the weakest AUC.
- **LightGBM** substantially outperforms both baselines by capturing nonlinear interactions, using all 450+ features including high-missing V-features, and handling NaN natively.
- **Identity features** (`id_*`, `DeviceType`, `DeviceInfo`) provide measurable lift over transaction-only features, as shown by the ablation study.
- **Chronological evaluation** (train on past ' evaluate on future) reflects the real deployment scenario; numbers are lower than a random-split evaluation would produce, but they are the honest ones.